# Analytical Solution for Groundwater Response to Sea-Level Rise

This notebook provides an analytical solution for groundwater response to sea-level rise (SLR) in a sloping coastal aquifer. The calculations are based on the work of Morgan & Werner (2016), who commented on Chesnaux (2015). The example is from the Pioneer Valley, Australia.

**References:**
- Morgan, L. K. and A. D. Werner (2016). "Comment on “Closed-form analytical solutions for assessing the consequences of sea-level rise on groundwater resources in sloping coastal aquifers”: paper published in Hydrogeology Journal (2015) 23:1399–1413, by R. Chesnaux." Hydrogeology journal 24(5): 1325-1328.
- Werner & Gallagher (2006). "Characterisation of sea-water intrusion in the Pioneer Valley, Australia using hydrochemistry and three-dim."

## 1. Imports and Setup

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## 2. Aquifer and Scenario Parameters

Here we define the key parameters for the aquifer, observation well, and the sea-level rise scenario.

In [ ]:
# --- Default Aquifer Parameters ---
DEFAULT_K = 100         # Hydraulic conductivity (m/d)
DEFAULT_Z0 = 25         # Aquifer thickness (m)
DEFAULT_RHO_F = 1000    # Density of freshwater (kg/m^3)
DEFAULT_RHO_S = 1025    # Density of saltwater (kg/m^3)
DEFAULT_W = 0.11/365.25 # Net recharge (m/d)

# --- Default Observation Well Data ---
DEFAULT_HB = 2          # Hydraulic head at observation well (m above MSL)
DEFAULT_XB = 2000       # Distance of observation well from the coast (m)

# --- Default Sea Level Rise Scenario ---
DEFAULT_DELTA_Z = 1.0     # Sea Level Rise (m)

## 3. Core Calculation Functions

These functions perform the core hydrogeological calculations based on the provided parameters.

In [ ]:
def calculate_delt(rho_f, rho_s):
    """Calculates the density difference ratio."""
    return (rho_s - rho_f) / rho_f

def calculate_coastal_flow(hb, xb, z0, k, w, delt):
    """Calculates the lateral groundwater flow at the coast (q0)."""
    qb = ((((hb + z0)**2 - ((1 + delt) * z0**2)) * k) - (w * xb**2)) / (2 * xb)
    q0 = qb + (w * xb)
    return q0

def calculate_groundwater_divide(q0, w):
    """Calculates the distance from the coast to the groundwater divide."""
    if w == 0:
        return np.inf
    return q0 / w

def calculate_mixed_convection_ratio(k, delt, z0, w, xn):
    """Calculates the mixed convection ratio (M), a measure of SWI vulnerability."""
    if w <= 0 or xn <= 0 or not np.isfinite(xn):
        return np.inf
    return k * delt * (1 + delt) * z0**2 / (w * xn**2)

def calculate_interface_toe_position(xn, m):
    """Calculates the position of the saltwater interface toe."""
    if 1 - m < 0 or not np.isfinite(m):
        return xn # Toe is at the divide if M >= 1
    return xn * (1 - np.sqrt(1 - m))

def calculate_head_at_x(x, q0, w, k, z0, delt):
    """Calculates the hydraulic head at a distance x from the coast."""
    inside_sqrt = (2 * q0 * x - w * x**2) / k + (1 + delt) * z0**2
    # Replace negative values with nan to avoid domain errors in sqrt
    with np.errstate(invalid='ignore'):
        head = np.sqrt(inside_sqrt) - z0
    return head

## 4. Visualization and Interactive Simulation

This section combines the calculations and plotting into an interactive tool. Use the sliders to change the input parameters and observe the effect on the saltwater intrusion toe and the aquifer cross-section.

In [ ]:
def plot_interactive_aquifer(k, z0, w, delta_z, hb, xb):
    """Function to run simulation and plot results for the interactive widget."""
    
    # --- Calculations ---
    delt = calculate_delt(DEFAULT_RHO_F, DEFAULT_RHO_S)
    
    # Pre-SLR
    q0_pre = calculate_coastal_flow(hb, xb, z0, k, w, delt)
    xn_pre = calculate_groundwater_divide(q0_pre, w)
    m_pre = calculate_mixed_convection_ratio(k, delt, z0, w, xn_pre)
    xt_pre = calculate_interface_toe_position(xn_pre, m_pre)
    
    # Post-SLR Head-Controlled
    z0_post_slr = z0 + delta_z
    qb_post_slr = ((((hb + z0)**2 - ((1 + delt) * z0_post_slr**2)) * k) - (w * xb**2)) / (2 * xb)
    q0_post_head = qb_post_slr + (w * xb)
    xn_post_head = calculate_groundwater_divide(q0_post_head, w)
    m_post_head = calculate_mixed_convection_ratio(k, delt, z0_post_slr, w, xn_post_head)
    xt_post_head = calculate_interface_toe_position(xn_post_head, m_post_head)

    # --- Plotting ---
    # Determine plot range
    max_xn = max(xn_pre, xn_post_head) if np.isfinite(xn_pre) and np.isfinite(xn_post_head) else 5000
    x = np.linspace(0, max_xn, 500)
    
    h_pre = calculate_head_at_x(x, q0_pre, w, k, z0, delt)
    h_post_head = calculate_head_at_x(x, q0_post_head, w, k, z0_post_slr, delt)

    fig, ax = plt.subplots(1, 1, figsize=(12, 6))

    # Plot Pre-SLR
    ax.plot(x, h_pre, 'b-', label=f'Piezometric Head (Pre-SLR)')
    ax.fill_between(x, -z0, h_pre, color='lightblue', alpha=0.5)
    ax.axvline(xt_pre, color='b', linestyle='--', label=f'Toe Pre-SLR: {xt_pre:.2f} m')

    # Plot Post-SLR
    ax.plot(x, h_post_head + delta_z, 'r-', label=f'Piezometric Head (Post-SLR)')
    ax.fill_between(x, -z0_post_slr, h_post_head + delta_z, color='lightcoral', alpha=0.4)
    ax.axvline(xt_post_head, color='r', linestyle='--', label=f'Toe Post-SLR: {xt_post_head:.2f} m')

    # Aquifer base and sea levels
    ax.axhline(y=0, color='blue', linestyle=':', label='Original MSL')
    ax.axhline(y=delta_z, color='red', linestyle=':', label=f'New MSL (+{delta_z}m)')
    ax.axhline(y=-z0, color='k', linestyle='-', label='Aquifer Base')

    ax.set_title('Interactive Aquifer Cross-Section (Head-Controlled Scenario)')
    ax.set_xlabel('Distance from Coast (m)')
    ax.set_ylabel('Elevation (m)')
    ax.legend(loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xlim(0, max_xn)
    ax.set_ylim(-z0 - 5, max(np.nanmax(h_pre), np.nanmax(h_post_head + delta_z)) + 5 if np.any(np.isfinite(h_pre)) else 10)
    
    plt.show()

    print(f"Pre-SLR Toe Position: {xt_pre:.2f} m")
    print(f"Post-SLR Toe Position (Head-Controlled): {xt_post_head:.2f} m")
    print(f"Landward Migration of Toe: {(xt_post_head - xt_pre):.2f} m")

In [ ]:
# Create widgets
k_slider = widgets.FloatSlider(value=DEFAULT_K, min=1, max=500, step=1, description='K (m/d):')
z0_slider = widgets.FloatSlider(value=DEFAULT_Z0, min=5, max=50, step=1, description='z0 (m):')
w_slider = widgets.FloatSlider(value=DEFAULT_W, min=0, max=0.001, step=0.00005, description='Recharge (m/d):', readout_format='.5f')
delta_z_slider = widgets.FloatSlider(value=DEFAULT_DELTA_Z, min=0, max=5, step=0.1, description='SLR (m):')
hb_slider = widgets.FloatSlider(value=DEFAULT_HB, min=0, max=10, step=0.5, description='Inland Head (m):')
xb_slider = widgets.FloatSlider(value=DEFAULT_XB, min=500, max=5000, step=100, description='Inland Distance (m):')

# Link widgets to the function
interactive_plot = widgets.interactive(plot_interactive_aquifer, 
                                       k=k_slider, 
                                       z0=z0_slider, 
                                       w=w_slider, 
                                       delta_z=delta_z_slider,
                                       hb=hb_slider,
                                       xb=xb_slider)

# Display the widgets and plot
display(interactive_plot)